# Taller final APO III: clasificación de calidad de frutas

Este notebook implementa una solución base para el caso de estudio del README usando tres categorías finales: **bueno**, **regular** y **malo**.

## CRISP-DM
1. Comprensión del negocio (Sección 1)
2. Comprensión de los datos (Sección 2)
3. Verificación de balance (Sección 3) + Preparación de los datos (Sección 4)
4. Vectorización y Modelado (Secciones 5, 6, 8)
5. Evaluación (Secciones 3, 7)
6. Despliegue / uso del modelo (Secciones 9, 10)

## Estructura esperada de los datos
Se espera una carpeta con subcarpetas por clase:

- `data/bueno/`
- `data/regular/`
- `data/malo/`

Cada subcarpeta debe contener imágenes `.jpg`, `.png` o `.jpeg`.

In [ ]:
import os, ctypes, glob, sys
venv_base = os.path.dirname(os.path.dirname(sys.executable))
nvidia_base = os.path.join(venv_base, "lib/python3.12/site-packages/nvidia")
if os.path.exists(nvidia_base):
    lib_paths = []
    for root, dirs, files in os.walk(nvidia_base):
        if root.endswith("/lib"):
            lib_paths.append(root)
    os.environ["LD_LIBRARY_PATH"] = ":".join(lib_paths)
    for f in glob.glob(os.path.join(nvidia_base, "**/lib*.so*"), recursive=True):
        if not f.endswith(".a"):
            try:
                ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
            except Exception:
                pass
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
import tensorflow as tf
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
import subprocess, sys, importlib, pkgutil, os
required = ["numpy", "pandas", "matplotlib", "seaborn", "scikit-learn", "Pillow", "tensorflow", "joblib"]
missing = [p for p in required if not importlib.util.find_spec(p)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from IPython.display import display
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import joblib

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["image.cmap"] = "gray"

try:
    import tensorflow as tf
    from tensorflow import keras
except ImportError:
    raise RuntimeError(
        "TensorFlow es obligatorio para la CNN. "
        "Asegúrate de usar Python 3.10-3.12 y ejecutar: pip install tensorflow"
    )

## 1. Comprensión del negocio

El objetivo es automatizar la clasificación visual de frutas para apoyar una línea de empaque o un sistema de inspección básica.

Las categorías finales del proyecto son:
- bueno
- regular
- malo

Estas etiquetas pueden representar fruta visualmente apta, con defectos leves o con deterioro evidente.

In [ ]:
# Ruta base del dataset. Ajusta esta variable si tus imágenes están en otro lugar.
DATA_DIR = Path("data")
CLASS_NAMES = ["bueno", "regular", "malo"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if not DATA_DIR.exists():
    print(f"La carpeta {DATA_DIR.resolve()} no existe todavía.")
    print("Crea la estructura data/bueno, data/regular y data/malo, o cambia DATA_DIR.")

rows = []
if DATA_DIR.exists():
    for class_name in CLASS_NAMES:
        class_dir = DATA_DIR / class_name
        if class_dir.exists():
            for file_path in class_dir.rglob("*"):
                if file_path.suffix.lower() in IMAGE_EXTENSIONS:
                    rows.append({"filepath": str(file_path), "label": class_name})

images_df = pd.DataFrame(rows)
print(f"Imágenes encontradas: {len(images_df)}")
images_df.head()

if not images_df.empty:
    display(images_df.sample(min(5, len(images_df)), random_state=42))
    print(images_df["label"].value_counts())

## 2. Comprensión de los datos

Se revisa la distribución de clases y algunos ejemplos visuales para verificar variabilidad, fondo, iluminación y calidad general de las imágenes.

In [ ]:
if not images_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, class_name in zip(axes, CLASS_NAMES):
        subset = images_df[images_df["label"] == class_name]
        if subset.empty:
            ax.axis("off")
            ax.set_title(f"Sin imágenes: {class_name}")
            continue
        sample_path = subset.sample(1, random_state=42)["filepath"].iloc[0]
        img = Image.open(sample_path).convert("RGB")
        ax.imshow(img)
        ax.set_title(class_name)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    counts = images_df["label"].value_counts().reindex(CLASS_NAMES, fill_value=0)
    counts.plot(kind="bar", color=["#4caf50", "#ffb300", "#e53935"])
    plt.title("Distribución de clases")
    plt.xlabel("Clase")
    plt.ylabel("Cantidad de imágenes")
    plt.show()

## 3. Verificación de balance de clases

Se verifica que las clases estén balanceadas. Si hay desbalanceo severo, se recomienda usar `class_weight` (ya configurado) o técnicas de sobremuestreo como SMOTE.

In [ ]:
if not images_df.empty:
    counts = images_df["label"].value_counts()
    print("Distribución de clases:")
    for cls in CLASS_NAMES:
        count = counts.get(cls, 0)
        pct = count / len(images_df) * 100
        print(f"  {cls}: {count} ({pct:.1f}%)")
    min_count = counts.min()
    max_count = counts.max()
    ratio = max_count / min_count if min_count > 0 else float("inf")
    print(f"\nRelación mayor/menor: {ratio:.1f}x")
    if ratio > 2.0:
        print("⚠️  Desbalanceo significativo. Considera aplicar SMOTE o aumentar datos de clases minoritarias.")
    else:
        print("✅ Clases razonablemente balanceadas.")

### Balanceo por subcarpeta de fruta

Si hay subcarpetas por tipo de fruta (ej. `Pomegranate_Good` con 5940 imágenes), se reduce cada subcarpeta a 1.5x la mediana para evitar que una fruta domine la clase.

In [ ]:
if not images_df.empty:
    parent_dirs = set(Path(p).parent.name for p in images_df["filepath"])
    if parent_dirs:
        for cls in CLASS_NAMES:
            cls_mask = images_df["label"] == cls
            sub_names = images_df.loc[cls_mask, "filepath"].apply(lambda p: Path(p).parent.name)
            sub_counts = sub_names.value_counts()
            if len(sub_counts) <= 1:
                continue
            median_count = int(sub_counts.median())
            max_allowed = max(int(median_count * 1.5), 1)
            keep = []
            for sub_name in sub_counts.index:
                sub_mask = cls_mask & (images_df["filepath"].apply(lambda p: Path(p).parent.name) == sub_name)
                subset = images_df[sub_mask]
                if len(subset) > max_allowed:
                    subset = subset.sample(n=max_allowed, random_state=42)
                keep.append(subset)
            images_df = pd.concat([images_df[~cls_mask]] + keep)

    print(f"Imágenes después de balancear por fruta: {len(images_df)}")
    counts = images_df["label"].value_counts()
    for cls in CLASS_NAMES:
        print(f"  {cls}: {counts.get(cls, 0)}")

## 4. Preparación de los datos

Se dividen los datos en entrenamiento, validación y prueba, manteniendo la proporción de clases con `stratify`.

In [ ]:
if images_df.empty:
    raise ValueError("No se encontraron imágenes. Agrega datos a data/bueno, data/regular y data/malo.")

X_paths = images_df["filepath"].to_numpy()
y_labels = images_df["label"].to_numpy()

X_train_paths, X_temp_paths, y_train, y_temp = train_test_split(
    X_paths,
    y_labels,
    test_size=0.30,
    random_state=42,
    stratify=y_labels,
)

X_val_paths, X_test_paths, y_val, y_test = train_test_split(
    X_temp_paths,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

print("Tamaño train:", len(X_train_paths))
print("Tamaño val:", len(X_val_paths))
print("Tamaño test:", len(X_test_paths))
print("Distribución train:\n", pd.Series(y_train).value_counts())

## 5. Vectorización de imágenes para modelos tradicionales

Para los modelos clásicos se extraen características simples a partir de la imagen: píxeles reescalados (128×128) y histogramas de color.

In [ ]:
IMG_SIZE = (128, 128)
HIST_BINS = 32


def resize_with_padding(image, target_size, bg_color=(255, 255, 255)):
    original_width, original_height = image.size
    aspect_ratio = original_width / original_height
    target_width, target_height = target_size
    if aspect_ratio > 1:
        new_width = target_width
        new_height = int(target_width / aspect_ratio)
    else:
        new_height = target_height
        new_width = int(target_height * aspect_ratio)
    image_resized = image.resize((new_width, new_height), Image.Resampling.LANCZOS)
    final_image = Image.new("RGB", target_size, bg_color)
    offset_x = (target_width - new_width) // 2
    offset_y = (target_height - new_height) // 2
    final_image.paste(image_resized, (offset_x, offset_y))
    return final_image


def load_image_array(path, size=IMG_SIZE):
    image = Image.open(path).convert("RGB")
    image = resize_with_padding(image, size)
    return np.asarray(image, dtype=np.float32) / 255.0


def extract_features(path):
    arr = load_image_array(path)
    hist_features = []
    for channel in range(3):
        channel_values = arr[:, :, channel].ravel()
        hist, _ = np.histogram(channel_values, bins=HIST_BINS, range=(0.0, 1.0))
        hist_features.append(hist)
    mean = arr.mean(axis=(0, 1))
    std = arr.std(axis=(0, 1))
    return np.concatenate([np.concatenate(hist_features), mean, std])


X_train_feat = np.vstack([extract_features(p) for p in X_train_paths])
X_val_feat = np.vstack([extract_features(p) for p in X_val_paths])
X_test_feat = np.vstack([extract_features(p) for p in X_test_paths])

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

print("Forma de X_train_feat:", X_train_feat.shape)
print("Clases:", list(label_encoder.classes_))

## 6. Modelado con aprendizaje tradicional

Se entrenan y comparan tres clasificadores clásicos sobre las características extraídas de las imágenes: **Logistic Regression**, **Random Forest** y **SVM**.

In [ ]:
models = {
    "logistic_regression": GridSearchCV(
        Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000)),
            ]
        ),
        param_grid={
            "clf__C": [0.1, 1.0, 10.0],
        },
        scoring="f1_macro",
        cv=3,
        n_jobs=-1,
        verbose=1,
    ),
    "random_forest": GridSearchCV(
        RandomForestClassifier(class_weight="balanced", random_state=42),
        param_grid={
            "n_estimators": [100, 200],
            "max_depth": [None, 20],
        },
        scoring="f1_macro",
        cv=3,
        n_jobs=-1,
        verbose=1,
    ),
    "svm": GridSearchCV(
        Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("clf", SVC(class_weight="balanced", probability=True, random_state=42)),
            ]
        ),
        param_grid={
            "clf__C": [0.5, 1.0],
            "clf__kernel": ["linear"],
        },
        scoring="f1_macro",
        cv=3,
        n_jobs=-1,
        verbose=1,
    ),
}

best_model_name = None
best_model = None
best_val_score = -np.inf
results_val = []

for name, search in models.items():
    print(f"Entrenando {name}...")
    search.fit(X_train_feat, y_train_enc)
    val_pred = search.predict(X_val_feat)
    val_f1 = f1_score(y_val_enc, val_pred, average="macro")
    results_val.append({"modelo": name, "val_f1_macro": val_f1})
    print(f"{name} -> val_f1_macro = {val_f1:.4f}")
    if val_f1 > best_val_score:
        best_val_score = val_f1
        best_model_name = name
        best_model = search.best_estimator_

results_val_df = pd.DataFrame(results_val).sort_values("val_f1_macro", ascending=False)
display(results_val_df)
print("Mejor modelo:", best_model_name)
print("Mejor score de validación:", best_val_score)

## 7. Evaluación del mejor modelo tradicional

Se reportan métricas sobre el conjunto de prueba y una matriz de confusión para identificar errores frecuentes.

In [ ]:
test_pred = best_model.predict(X_test_feat)

print("Accuracy:", accuracy_score(y_test_enc, test_pred))
print("Precision macro:", precision_score(y_test_enc, test_pred, average="macro", zero_division=0))
print("Recall macro:", recall_score(y_test_enc, test_pred, average="macro", zero_division=0))
print("F1 macro:", f1_score(y_test_enc, test_pred, average="macro"))
print("F1 micro:", f1_score(y_test_enc, test_pred, average="micro"))

report = classification_report(y_test_enc, test_pred, target_names=label_encoder.classes_, zero_division=0)
print(report)

cm = confusion_matrix(y_test_enc, test_pred)
cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
plt.figure(figsize=(7, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title(f"Matriz de confusión - {best_model_name}")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.show()

errors_df = pd.DataFrame({
    "filepath": X_test_paths,
    "y_true": label_encoder.inverse_transform(y_test_enc),
    "y_pred": label_encoder.inverse_transform(test_pred),
})
errors_df = errors_df[errors_df["y_true"] != errors_df["y_pred"]]
print("Ejemplos mal clasificados:")
display(errors_df.head(10))

## 8. Modelado con deep learning

Si TensorFlow está disponible, se entrena una CNN sobre las imágenes redimensionadas a 128×128 con data augmentation.

In [ ]:
cnn_history = None
cnn_model = None
cnn_test_metrics = None

if tf is None:
    print("TensorFlow no está disponible, se omite CNN.")
else:
    image_size = (64, 64)
    batch_size = 8

    def decode_image(path, label):
        image = tf.io.read_file(path)
        image = tf.image.decode_image(image, channels=3, expand_animations=False)
        image = tf.image.resize(image, image_size)
        image = tf.cast(image, tf.float32) / 255.0
        label = tf.one_hot(label, depth=len(label_encoder.classes_))
        return image, label

    def augment(image, label):
        image = tf.image.random_flip_left_right(image, seed=42)
        image = tf.image.random_brightness(image, 0.15, seed=42)
        image = tf.image.random_contrast(image, 0.9, 1.1, seed=42)
        return image, label

    train_ds = tf.data.Dataset.from_tensor_slices((X_train_paths, y_train_enc))
    train_ds = train_ds.shuffle(buffer_size=len(X_train_paths), seed=42)
    train_ds = train_ds.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    val_ds = tf.data.Dataset.from_tensor_slices((X_val_paths, y_val_enc))
    val_ds = val_ds.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    test_ds = tf.data.Dataset.from_tensor_slices((X_test_paths, y_test_enc))
    test_ds = test_ds.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)
    test_ds = test_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    cnn_model = keras.Sequential([
        keras.layers.Input(shape=(image_size[0], image_size[1], 3)),
        keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        keras.layers.BatchNormalization(),
        keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        keras.layers.BatchNormalization(),
        keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
        keras.layers.BatchNormalization(),
        keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPooling2D(),
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(len(label_encoder.classes_), activation="softmax"),
    ])

    cnn_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=5e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6),
    ]

    cnn_history = cnn_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=50,
        callbacks=callbacks,
        verbose=1,
    )

    cnn_test_metrics = cnn_model.evaluate(test_ds, verbose=0)
    print(f"CNN - Test loss: {cnn_test_metrics[0]:.4f}, Test accuracy: {cnn_test_metrics[1]:.4f}")


## 9. Despliegue y guardado de artefactos

Se guardan el mejor modelo tradicional y, si existe, la CNN entrenada para reutilizarlos después en una interfaz o script de inferencia.

In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

joblib.dump(best_model, OUTPUT_DIR / "best_traditional_model.pkl")
joblib.dump(label_encoder, OUTPUT_DIR / "label_encoder.pkl")
joblib.dump({
    "image_size": IMG_SIZE,
    "hist_bins": HIST_BINS,
    "class_names": CLASS_NAMES,
}, OUTPUT_DIR / "feature_config.pkl")

if cnn_model is not None and tf is not None:
    cnn_model.save(OUTPUT_DIR / "cnn_model.keras")

print(f"Artefactos guardados en: {OUTPUT_DIR.resolve()}")
print("Para guardar el notebook, usa File -> Save en VS Code o guarda el archivo .ipynb actual.")


def predict_image(image_path):
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(image_path)
    features = extract_features(image_path)
    pred_encoded = best_model.predict([features])[0]
    pred_label = label_encoder.inverse_transform([pred_encoded])[0]
    return pred_label


# Ejemplo de uso:
# predict_image("data/regular/ejemplo.jpg")

## 10. Inferencia en nueva imagen

Esta celda permite realizar predicciones sobre una imagen nueva usando el mejor modelo guardado.

## 11. Metodología CRISP-DM aplicada

```
┌─────────────────────────────────────────────────────────────┐
│             1. COMPRENSIÓN DEL NEGOCIO                      │
│  • Clasificar calidad de frutas (bueno/regular/malo)       │
│  • Impacto: reducir desperdicio, estandarizar calidad      │
└──────────────────────┬──────────────────────────────────────┘
                       ▼
┌─────────────────────────────────────────────────────────────┐
│             2. COMPRENSIÓN DE LOS DATOS                     │
│  • 23,064 imágenes (11664 bueno, 4612 regular, 6788 malo)  │
│  • Fuentes: Kaggle Fruit Quality + recolección propia       │
│  • Desbalanceo 2.5x entre clase mayor y menor              │
└──────────────────────┬──────────────────────────────────────┘
                       ▼
┌─────────────────────────────────────────────────────────────┐
│             3. PREPARACIÓN DE LOS DATOS                     │
│  • Redimensionamiento con padding (128×128)                │
│  • Extracción: histogramas de color + media/std            │
│  • Balanceo por subcarpeta de fruta                        │
│  • División train/val/test (70/15/15) estratificada        │
└──────────────────────┬──────────────────────────────────────┘
                       ▼
┌─────────────────────────────────────────────────────────────┐
│             4. MODELADO                                     │
│  • Clásicos: Logistic Regression, Random Forest, SVM       │
│  • Deep Learning: CNN (3 bloques Conv + BatchNorm)         │
│  • GridSearchCV con validación cruzada (3-fold)            │
└──────────────────────┬──────────────────────────────────────┘
                       ▼
┌─────────────────────────────────────────────────────────────┐
│             5. EVALUACIÓN                                   │
│  • Métricas: Accuracy, Precision, Recall, F1 (macro/micro) │
│  • Matriz de confusión                                     │
│  • Comparación entre modelos clásicos y CNN                │
└──────────────────────┬──────────────────────────────────────┘
                       ▼
┌─────────────────────────────────────────────────────────────┐
│             6. DESPLIEGUE                                   │
│  • Modelos guardados en outputs/                           │
│  • Interfaz Streamlit (app/app.py)                         │
│  • Función predict_image para inferencia                   │
└─────────────────────────────────────────────────────────────┘
```

## 12. Arquitectura del sistema

```
┌──────────────┐    ┌────────────────────┐    ┌──────────────────┐
│   Entrada     │    │   Preprocesamiento  │    │    Modelos        │
│              │    │                    │    │                  │
│  Cámara web  │───▶│  Redimensionar     │───▶│  Random Forest   │
│  o archivo   │    │  Padding (128×128) │    │  CNN             │
│              │    │  Normalizar        │    │                  │
└──────────────┘    └────────────────────┘    └────────┬─────────┘
                                                       ▼
┌──────────────────────────────────────────────────────────────┐
│                    Salida (Streamlit UI)                     │
│  • Calidad: Bueno / Regular / Malo                          │
│  • Confianza de predicción (%)                              │
└──────────────────────────────────────────────────────────────┘
```

## 13. Análisis de aspectos éticos

### 13.1 Sesgo en los datos
- El dataset contiene desbalanceo natural (más frutas "buenas" que "malas"), mitigado con balanceo por subcarpeta.
- Las imágenes provienen de Kaggle (contexto internacional) más recolección local, lo que puede introducir variaciones.

### 13.2 Privacidad y datos
- Las imágenes son de frutas, no de personas. No aplican restricciones de datos personales.

### 13.3 Impacto laboral
- La automatización puede complementar el trabajo de inspectores, reduciendo errores y fatiga.

### 13.4 Transparencia
- Los modelos Random Forest y Logistic Regression son interpretables mediante coeficientes e importancia de características.

### 13.5 Reproducibilidad
- Semillas aleatorias fijas (`random_state=42`) en todos los modelos.

## 14. Análisis de impactos de la solución

### 14.1 Impacto económico
- Reducción de desperdicio, estandarización de calidad, velocidad de procesamiento.

### 14.2 Impacto social
- Acceso a tecnología de clasificación para pequeños agricultores.
- Interfaz web funciona sin hardware especializado.

### 14.3 Impacto ambiental
- Reducción de huella de carbono al disminuir desperdicio de alimentos.

### 14.4 Limitaciones
- Requiere fondo simple y uniforme.
- La CNN actualmente tiene menor precisión que Random Forest.

## 15. Trabajo futuro

1. Mejorar CNN con transfer learning.
2. Expandir clases a más frutas y verduras.
3. Implementar en Raspberry Pi para simulación en tiempo real.
4. Desplegar app con TensorFlow Lite.
5. Grabar video demostrativo del proyecto.

## 16. Referencias

[1] R. Park, "Fruit Quality Classification," Kaggle, 2023. [Online]. Available: https://www.kaggle.com/datasets/ryandpark/fruit-quality-classification

[2] A. Géron, *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd ed. O'Reilly Media, 2022.

[3] F. Chollet, *Deep Learning with Python*, 2nd ed. Manning Publications, 2021.

[4] L. Breiman, "Random Forests," *Machine Learning*, vol. 45, no. 1, pp. 5–32, 2001.

[5] Imágenes recolectadas por el grupo de trabajo (2026) en plazas de mercado locales.